# 🚀 ConveyorVision — YOLO26 Training & Visual Analytics
**Hệ thống phân loại sản phẩm đạt chuẩn (`pass: 0`) và phát hiện lỗi (`defect: 1`) trên băng chuyền công nghiệp.**

Notebook này được thiết kế tối ưu để:
- Chạy liền mạch trên cả **Google Colab GPU** (thông qua Colab extension trong Antigravity IDE hoặc trình duyệt) và **Máy tính cục bộ (Local)**.
- Tự động nhận diện môi trường làm việc, hỗ trợ đồng bộ mã nguồn và dataset chuẩn 2 lớp (`pass` và `defect`).
- Tích hợp cơ chế **Checkpoint Resume** (`last.pt`) có **xác thực kiến trúc YOLO26** chống gián đoạn (nếu mất điện hoặc Colab ngắt kết nối).
- Tích hợp kiểm tra mã thoát (exit code) nghiêm ngặt nhằm tránh hiện tượng báo thành công giả.
- Tích hợp hệ thống biểu đồ phân tích trực quan toàn diện (Loss curves, mAP@50, mAP@50-95, Normalized Confusion Matrix, PR Curve và ảnh dự đoán thực tế).

## 1. Kiểm tra Môi Trường & Cấu Hình Phần Cứng (GPU/CPU)
> [!IMPORTANT]
> - Huấn luyện YOLO trên **GPU (NVIDIA T4)** chỉ mất khoảng **10 - 15 phút** cho 30 epochs.
> - Nếu kết quả bên dưới hiển thị **CPU**, bạn nên đổi sang GPU trên Colab: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU** để tăng tốc x15 lần!

In [1]:
import os
import sys
import torch

print("=" * 65)
print("🔍 KIỂM TRA MÔI TRƯỜNG & PHẦN CỨNG HỆ THỐNG")
print("=" * 65)
print(f"🐍 Python Version : {sys.version.split()[0]}")
print(f"🔥 PyTorch Version: {torch.__version__}")

is_cuda = torch.cuda.is_available()
print(f"⚡ CUDA Khả Dụng  : {is_cuda}")

if is_cuda:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🚀 Thiết bị GPU   : {gpu_name} ({gpu_mem} GB VRAM)")
    print("✨ Sẵn sàng huấn luyện tốc độ cao với GPU!")
else:
    print("⚠️ CẢNH BÁO: Đang chạy trên CPU (chưa nhận diện GPU)!")
    print("👉 Hướng dẫn bật GPU:")
    print("   1. Nếu chạy trên Colab Web: Runtime -> Change runtime type -> T4 GPU.")
    print("   2. Nếu kết nối Colab từ Antigravity IDE: Chọn Connect -> New Connection -> GPU Runtime.")
print("=" * 65)


🔍 KIỂM TRA MÔI TRƯỜNG & PHẦN CỨNG HỆ THỐNG
🐍 Python Version : 3.13.16
🔥 PyTorch Version: 2.11.0+cpu
⚡ CUDA Khả Dụng  : False
⚠️ CẢNH BÁO: Đang chạy trên CPU (chưa nhận diện GPU)!
👉 Hướng dẫn bật GPU:
   1. Nếu chạy trên Colab Web: Runtime -> Change runtime type -> T4 GPU.
   2. Nếu kết nối Colab từ Antigravity IDE: Chọn Connect -> New Connection -> GPU Runtime.


## 2. Cài Đặt Các Thư Viện Cần Thiết
Cài đặt Ultralytics YOLO26 (>=8.4.0), Roboflow và các thư viện trực quan hóa dữ liệu.

In [2]:
%pip install -q "ultralytics>=8.4.0" matplotlib seaborn pandas pillow roboflow pyyaml


## 3. Đồng Bộ Thư Mục Làm Việc & Chuẩn Bị Dữ Liệu
Cell này tự động:
1. Phát hiện môi trường (Google Colab vs Máy cục bộ) và thiết lập thư mục làm việc.
2. Tự động xử lý clone dự án hoặc khởi tạo cấu trúc thư mục độc lập (tránh lỗi khi repo riêng tư).
3. Tự động tạo và cấu hình file `data/data.yaml` với đường dẫn tuyệt đối chuẩn xác cho môi trường hiện tại.
4. Kiểm tra mã thoát (exit code) từ `dataset_manager.py` — nếu phát sinh lỗi sẽ chặn và dừng ngay lập tức, không báo thành công giả.
5. Đọc Roboflow API key an toàn qua biến môi trường `ROBOFLOW_API_KEY` hoặc Secrets, không lưu cứng key trong mã nguồn.

In [3]:
import os
import sys
import shutil
from pathlib import Path

print("=" * 65)
print("📂 ĐỒNG BỘ THƯ MỤC LÀM VIỆC & BỘ DỮ LIỆU CONVEYORVISION")
print("=" * 65)

# 1. Nhận diện môi trường & thiết lập Working Directory chuẩn
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    print("🌐 Môi trường: Google Colab Remote Runtime")
    repo_dir = Path("/content/ConveyorVision")
    if not repo_dir.exists():
        print("📥 Đang chuẩn bị không gian làm việc tại /content/ConveyorVision...")
        clone_success = False
        try:
            gh_token = os.getenv("GITHUB_TOKEN", "").strip()
            if gh_token:
                res = os.system(f"git clone https://{gh_token}@github.com/anky06-ky/ConveyorVision.git /content/ConveyorVision")
                clone_success = (res == 0 and repo_dir.exists())
            if not clone_success:
                res = os.system("git clone https://github.com/anky06-ky/ConveyorVision.git /content/ConveyorVision 2>/dev/null")
                clone_success = (res == 0 and repo_dir.exists())
        except Exception:
            clone_success = False

        if not clone_success:
            print("ℹ️ Chạy ở chế độ Colab Standalone (tạo workspace độc lập)...")
            repo_dir.mkdir(parents=True, exist_ok=True)

    os.chdir(repo_dir)
    project_root = repo_dir
else:
    print("💻 Môi trường: Local Machine")
    current = Path.cwd().resolve()
    if (current / "src").exists() and (current / "data").exists():
        project_root = current
    elif (current.parent / "src").exists() and (current.parent / "data").exists():
        project_root = current.parent
        os.chdir(project_root)
    else:
        project_root = current

print(f"📁 Thư mục làm việc hiện tại: {Path.cwd()}")

# 2. Đảm bảo cấu trúc thư mục đầy đủ
data_dir = (project_root / "data").resolve()
for split in ["train", "val", "test"]:
    (data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
(project_root / "models").mkdir(parents=True, exist_ok=True)
(project_root / "runs").mkdir(parents=True, exist_ok=True)

# 3. Tự động tạo & cấu hình data/data.yaml với đường dẫn tuyệt đối chính xác
data_yaml_path = data_dir / "data.yaml"
yaml_content = f"""# ConveyorVision YOLO Dataset Configuration (Auto-configured)
path: {data_dir.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: pass
  1: defect
"""

with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"✅ Đã cấu hình dataset YAML: {data_yaml_path}")

# 4. Kiểm tra số lượng ảnh hiện có trong data/images/train
train_imgs = list((data_dir / "images" / "train").glob("*.*"))
print(f"📊 Số lượng ảnh hiện có trong data/images/train: {len(train_imgs)}")

if len(train_imgs) == 0:
    print("⚠️ Dữ liệu ảnh chưa có trong data/. Đang chuẩn bị tải bộ dữ liệu qua Roboflow API...")
    ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
    if not ROBOFLOW_API_KEY and IS_COLAB:
        try:
            from google.colab import userdata
            ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
        except Exception:
            pass
    if not ROBOFLOW_API_KEY:
        import getpass
        print("🔑 Vui lòng nhập Roboflow API Key để tải dataset:")
        ROBOFLOW_API_KEY = getpass.getpass("Roboflow API Key: ").strip()

    if not ROBOFLOW_API_KEY:
        raise ValueError("❌ Lỗi: Chưa cung cấp Roboflow API Key! Vui lòng thiết lập biến môi trường ROBOFLOW_API_KEY.")

    mgr_script = project_root / "src" / "conveyorvision" / "dataset_manager.py"
    if mgr_script.exists():
        os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
        import subprocess
        # Thực thi dataset_manager qua subprocess.run để bắt chính xác returncode nguyên vẹn
        # (tránh lỗi IPython system() trả về None khiến điều kiện '!= 0' ném lỗi giả)
        res = subprocess.run(
            [sys.executable, str(mgr_script), "--root", str(project_root), "--download-all"],
            cwd=str(project_root),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print("⚠️ Lỗi từ dataset_manager.py (stderr):", file=sys.stderr)
            print(res.stderr, file=sys.stderr, end="")
        if res.returncode != 0:
            raise RuntimeError(f"❌ LỖI: Đồng bộ dataset bằng dataset_manager.py thất bại với exit code {res.returncode}! Vui lòng kiểm tra log lỗi phía trên.")
    else:
        from roboflow import Roboflow
        import yaml
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        target_datasets = [
            {"workspace": "anky06", "project": "conveyor-belt-eafpe-bgzzm", "version": 1, "prefix": "conveyor", "mapping": {"bottle": 0}},
            {"workspace": "bayo-et081", "project": "bottle-detection-qti1t", "version": 1, "prefix": "bayo", "mapping": {"defect": 1, "ok": 0}},
            {"workspace": "anky06", "project": "bottle_defects-tpmcq-kijtp", "version": 1, "prefix": "bottle_defect", "mapping": {"good": 0, "defective": 1}}
        ]
        raw_dir = project_root / "raw_datasets"
        raw_dir.mkdir(parents=True, exist_ok=True)

        def normalize_split(s):
            s = s.lower().strip()
            return "val" if s in ["valid", "validation", "val"] else ("test" if s in ["test", "testing"] else "train")

        dl_errors = []
        for ds in target_datasets:
            print(f"📦 Đang tải dataset: {ds['workspace']}/{ds['project']}...")
            try:
                dl = rf.workspace(ds["workspace"]).project(ds["project"]).version(ds["version"]).download("yolov8", location=str(raw_dir / ds["project"]))
                dl_path = Path(dl.location)
                orig_yaml = dl_path / "data.yaml"
                orig_names = {}
                if orig_yaml.exists():
                    with open(orig_yaml, "r", encoding="utf-8") as yf:
                        ydata = yaml.safe_load(yf) or {}
                        orig_names = ydata.get("names", {})
                        if isinstance(orig_names, list):
                            orig_names = {i: name for i, name in enumerate(orig_names)}

                id_map = {}
                for orig_id, orig_n in orig_names.items():
                    n_clean = str(orig_n).lower().strip()
                    if n_clean in ds["mapping"]:
                        id_map[int(orig_id)] = ds["mapping"][n_clean]
                    elif "defect" in n_clean or "damage" in n_clean:
                        id_map[int(orig_id)] = 1
                    else:
                        id_map[int(orig_id)] = 0

                for orig_s in ["train", "valid", "val", "test"]:
                    s_norm = normalize_split(orig_s)
                    src_img_dir = dl_path / orig_s / "images" if (dl_path / orig_s / "images").exists() else dl_path / orig_s
                    src_lbl_dir = dl_path / orig_s / "labels" if (dl_path / orig_s / "labels").exists() else None
                    if not src_img_dir.exists():
                        continue
                    for img_f in src_img_dir.glob("*.*"):
                        if img_f.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                            continue
                        dest_img = data_dir / "images" / s_norm / f"{ds['prefix']}_{img_f.name}"
                        dest_lbl = data_dir / "labels" / s_norm / f"{ds['prefix']}_{img_f.stem}.txt"
                        shutil.copy2(img_f, dest_img)
                        new_lines = []
                        orig_lbl_f = src_lbl_dir / f"{img_f.stem}.txt" if src_lbl_dir else None
                        if orig_lbl_f and orig_lbl_f.exists():
                            with open(orig_lbl_f, "r", encoding="utf-8") as lf:
                                for line in lf:
                                    parts = line.strip().split()
                                    if len(parts) >= 5:
                                        cid = int(parts[0])
                                        target_cid = id_map.get(cid, 0)
                                        new_lines.append(f"{target_cid} " + " ".join(parts[1:5]) + "\n")
                        with open(dest_lbl, "w", encoding="utf-8") as lf:
                            lf.writelines(new_lines)
            except Exception as e:
                dl_errors.append(f"{ds['project']}: {e}")
                print(f"⚠️ Không thể tải {ds['project']}: {e}")

        if dl_errors and len(dl_errors) == len(target_datasets):
            raise RuntimeError(f"❌ LỖI: Tất cả các dataset đều tải thất bại:\n" + "\n".join(dl_errors))

    train_imgs = list((data_dir / "images" / "train").glob("*.*"))
    if len(train_imgs) == 0:
        raise RuntimeError("❌ LỖI: Không tìm thấy ảnh nào trong data/images/train sau khi tải! Vui lòng kiểm tra lại cấu hình Roboflow.")
    print(f"✅ Đã đồng bộ hoàn tất: {len(train_imgs)} ảnh train sẵn sàng!")
else:
    print("🎉 Dữ liệu 2.666 ảnh đã có sẵn, sẵn sàng bước vào huấn luyện!")
print("=" * 65)


📂 ĐỒNG BỘ THƯ MỤC LÀM VIỆC & BỘ DỮ LIỆU CONVEYORVISION
🌐 Môi trường: Google Colab Remote Runtime
📁 Thư mục làm việc hiện tại: /content/ConveyorVision
✅ Đã cấu hình dataset YAML: /content/ConveyorVision/data/data.yaml
📊 Số lượng ảnh hiện có trong data/images/train: 0
⚠️ Dữ liệu ảnh chưa có trong data/. Đang chuẩn bị tải bộ dữ liệu qua Roboflow API...
🔑 Vui lòng nhập Roboflow API Key để tải dataset:
[OK] Cập nhật file cấu hình: /content/ConveyorVision/data/data.yaml

>>> Đang tải từ Roboflow: anky06/conveyor-belt-eafpe-bgzzm (version 1)...

loading Roboflow workspace...

loading Roboflow project...

Exporting format yolov8 in progress : 0.0%
Exporting format yolov8 in progress : 0.1%
Exporting format yolov8 in progress : 41.0%

Version export complete for yolov8 format

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=

⚠️ Lỗi từ dataset_manager.py (stderr):


Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::   0%|          | 0/3485 [00:00<?, ?it/s]
Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::  14%|█▍        | 493/3485 [00:00<00:00, 4923.14it/s]
Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::  28%|██▊       | 986/3485 [00:00<00:00, 4143.28it/s]
Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::  40%|████      | 1409/3485 [00:00<00:00, 3984.38it/s]
Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::  56%|█████▌    | 1938/3485 [00:00<00:00, 4453.63it/s]
Extracting Dataset Version Zip to /content/ConveyorVision/raw_datasets/conveyor-belt-eafpe-bgzzm in yolov8::  84%|████████▍ | 2923/3485 [00:00<00:00, 6287.34it

## 4. Huấn Luyện Mô Hình YOLO26 (30 Epochs) Có Tích Hợp Resume & Kiểm Tra Kiến Trúc
- Sử dụng mô hình `yolo26n.pt` nhẹ, tốc độ inference cao phù hợp băng chuyền công nghiệp (> 60 FPS trên GPU, > 12 FPS trên CPU).
- **Kiểm tra kiến trúc thông minh:** Tự động phát hiện nếu file `last.pt` là mô hình YOLOv8 cũ để ngăn chặn việc tiếp tục sai kiến trúc; chỉ resume khi đúng là checkpoint YOLO26 (`end2end: True` / `yolo26`).
- **Xử lý mất cân bằng lớp:** Với tỷ lệ 5.096 box `pass` so với 524 box `defect` (~10:1), thiết lập `cls=1.2` (mặc định Ultralytics là 0.5) giúp tăng cường trọng số hàm mất mát phân loại cho các mẫu chai lỗi.

In [4]:
import os
from pathlib import Path
from ultralytics import YOLO

# Đảm bảo đường dẫn chính xác
if 'project_root' in locals():
    os.chdir(project_root)
elif os.path.exists('/content/ConveyorVision'):
    os.chdir('/content/ConveyorVision')

yaml_path = Path("data/data.yaml").resolve()
assert yaml_path.exists(), f"Lỗi: Không tìm thấy file {yaml_path}. Hãy chạy Cell 3 trước!"

print(f"🎯 Sử dụng dataset config: {yaml_path}")

# Kiểm tra checkpoint lưu dở và xác thực kiến trúc YOLO26
ckpt_path = Path("runs/train/conveyor_yolo26_30/weights/last.pt")
can_resume = False

if ckpt_path.exists():
    print(f"🔍 Phát hiện checkpoint lưu dở tại: {ckpt_path}")
    try:
        candidate = YOLO(str(ckpt_path))
        train_args_model = str(candidate.ckpt.get("train_args", {}).get("model", "")).lower() if hasattr(candidate, "ckpt") and candidate.ckpt else ""
        yaml_file = str(candidate.model.yaml.get("yaml_file", "")).lower() if hasattr(candidate, "model") and hasattr(candidate.model, "yaml") and candidate.model.yaml else ""
        is_end2end = getattr(candidate.model, "yaml", {}).get("end2end", False) if hasattr(candidate, "model") else False

        if "yolo26" in train_args_model or "yolo26" in yaml_file or is_end2end:
            print(f"✅ Xác thực thành công: Checkpoint thuộc kiến trúc YOLO26 ({yaml_file or train_args_model or 'end2end'}).")
            can_resume = True
            model = candidate
        else:
            print(f"⚠️ CẢNH BÁO: Checkpoint tại {ckpt_path} KHÔNG PHẢI kiến trúc YOLO26 (phát hiện model='{train_args_model}', yaml='{yaml_file}')!")
            print("🚫 Để tránh xung đột với YOLOv8 cũ, hệ thống sẽ bỏ qua checkpoint này và bắt đầu huấn luyện mới với yolo26n.pt.")
            can_resume = False
    except Exception as e:
        print(f"⚠️ Không thể nạp checkpoint ({e}). Bắt đầu huấn luyện mới.")
        can_resume = False

if can_resume:
    print("▶️ Đang tiếp tục huấn luyện (RESUME)...\n")
    results = model.train(resume=True)
else:
    print("🚀 Bắt đầu huấn luyện mới với mô hình YOLO26 (yolo26n.pt - 30 Epochs)...")
    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(yaml_path),
        epochs=30,
        batch=16,
        imgsz=640,
        project="runs/train",
        name="conveyor_yolo26_30",
        exist_ok=True,
        plots=True,
        save=True,
        verbose=True,
        # Tham số hỗ trợ xử lý mất cân bằng lớp (5.096 pass vs 524 defect):
        cls=1.2           # Tăng trọng số loss classification cho class defect (mặc định Ultralytics là 0.5)
    )

print("🏆 QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT THÀNH CÔNG!")


🎯 Sử dụng dataset config: /content/ConveyorVision/data/data.yaml
🚀 Bắt đầu huấn luyện mới với mô hình YOLO26 (yolo26n.pt - 30 Epochs)...
Ultralytics 8.4.178 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.2, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ConveyorVision/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

KeyboardInterrupt: 

## 5. Trực Quan Hóa Đồ Thị & Chỉ Số Đánh Giá (Visual Analytics)
Hiển thị trực quan toàn diện:
1. **Tiến trình huấn luyện (Training Curves):** Box Loss, Class Loss, DFL Loss qua từng epoch; Precision, Recall, mAP50, mAP50-95.
2. **Ma trận nhầm lẫn (Normalized Confusion Matrix):** Đánh giá chi tiết tỷ lệ phân loại đúng giữa chai Đạt (`pass`) và chai Lỗi (`defect`).
3. **Đường cong Precision-Recall (PR Curve):** Đánh giá độ tin cậy của thuật toán tại mọi ngưỡng threshold.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Tìm thư mục kết quả huấn luyện
possible_dirs = [
    Path('runs/detect/runs/train/conveyor_yolo26_30'),
    Path('runs/train/conveyor_yolo26_30'),
    Path('.'),  # kết quả ZIP có thể đã giải nén ở thư mục gốc project
    Path('runs/train/conveyor_yolo30'),
    Path('runs/pilot/pilot_test'),
    Path('runs/detect/runs/pilot/pilot_test')
]
if Path('runs/train').exists():
    possible_dirs.extend(sorted([d for d in Path('runs/train').glob('*') if d.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True))

exp_dir = None
for d in possible_dirs:
    if d.exists() and ((d / 'results.png').exists() or (d / 'confusion_matrix.png').exists()):
        exp_dir = d
        break

if exp_dir is None:
    exp_dir = Path('runs/detect/runs/train/conveyor_yolo26_30')

print(f"📊 Đang hiển thị kết quả phân tích từ thư mục: {exp_dir.resolve()}")

# 1. Biểu đồ Loss & Metrics toàn bộ quá trình
results_img = exp_dir / 'results.png'
if results_img.exists():
    plt.figure(figsize=(16, 8), dpi=150)
    plt.imshow(Image.open(results_img))
    plt.title('Biểu Đồ Tiến Trình Huấn Luyện (Loss & Performance Metrics)', fontsize=15, fontweight='bold', pad=12)
    plt.axis('off')
    plt.show()
else:
    print("⚠️ Chưa tìm thấy results.png (mô hình cần chạy ít nhất 1 epoch để sinh biểu đồ)")

# 2. Ma trận nhầm lẫn (Confusion Matrix)
cm_img = exp_dir / 'confusion_matrix_normalized.png'
if not cm_img.exists():
    cm_img = exp_dir / 'confusion_matrix.png'

if cm_img.exists():
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(Image.open(cm_img))
    plt.title('Ma Trận Nhầm Lẫn Chuẩn Hóa (Normalized Confusion Matrix)', fontsize=14, fontweight='bold', pad=12)
    plt.axis('off')
    plt.show()

# 3. Đường cong Precision - Recall (BoxPR Curve)
pr_img = exp_dir / 'BoxPR_curve.png'
if pr_img.exists():
    plt.figure(figsize=(10, 7), dpi=150)
    plt.imshow(Image.open(pr_img))
    plt.title('Đường Cong Precision - Recall (PR Curve)', fontsize=14, fontweight='bold', pad=12)
    plt.axis('off')
    plt.show()


## 6. Trực Quan Hóa Ảnh Dự Đoán Thực Tế (Inference Preview)
Kiểm tra trực tiếp kết quả mô hình gán nhãn bounding box trên tập dữ liệu kiểm thử (Validation Batch) với nhãn `0: pass` (chai đạt) và `1: defect` (chai lỗi móp méo, vỡ, thiếu nắp...).

In [ ]:
# Hiển thị ảnh dự đoán bounding box trên tập validation
pred_images = sorted(list(exp_dir.glob('val_batch*_pred.jpg'))) + sorted(list(exp_dir.glob('val_batch*_pred.png')))

if pred_images:
    n_show = min(2, len(pred_images))
    fig, axes = plt.subplots(1, n_show, figsize=(18, 9), dpi=150)
    if n_show == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, pred_images[:n_show]):
        ax.imshow(Image.open(img_path))
        ax.set_title(f"Dự đoán trên tập Validation: {img_path.name}", fontsize=13, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Chưa có ảnh dự đoán trong thư mục kết quả. Ảnh sẽ được tự động sinh sau khi hoàn thành huấn luyện/đánh giá.")


## 7. Xuất Trọng Số Tối Ưu Ra Thư Mục Models
Lưu file trọng số tốt nhất (`best.pt`) vào `models/` để sẵn sàng tích hợp vào pipeline kiểm định băng chuyền thời gian thực (Camera Stream).

In [ ]:
import shutil

best_weight = exp_dir / "weights" / "best.pt"
target_weight = Path("models") / "conveyor_yolo26_30ep_best.pt"
target_weight.parent.mkdir(parents=True, exist_ok=True)

if best_weight.exists():
    shutil.copy2(best_weight, target_weight)
    size_mb = round(target_weight.stat().st_size / (1024 * 1024), 2)
    print(f"🎉 Đã lưu trọng số tối ưu thành công tại: {target_weight.resolve()} ({size_mb} MB)")
    
    if IS_COLAB:
        try:
            from google.colab import files
            print("⬇️ Đang khởi tạo tải trọng số về máy tính của bạn...")
            files.download(str(target_weight))
        except Exception:
            print(f"💡 Để tải về máy: Click chuột phải vào file '{target_weight.as_posix()}' trên thanh Files bên trái Colab -> Chọn Download.")
else:
    print(f"ℹ️ File trọng số {best_weight} chưa sẵn sàng.")
